# Stage 24 — TENTACLES strict consensus-vote sensitivity (5 vs 6)

**Purpose.** Using saved native `getConsensus()` vote tables, compare an adapted **n.min=5** consensus with the published **n.min=6** majority threshold among 9 voting algorithms (excluding XGBoost). There is **no native R retraining**, no new feature selection, and no held-out-label-driven tuning.

**Predictive endpoint.** For each held-out cohort, standardise the selected gene expression on the other three cohorts only, then fit the pre-established fixed balanced logistic regression (`L2`, `C=1`, `liblinear`; threshold `0.5`). Compute held-out metrics and paired bootstrapped descriptive sensitivity intervals. The previously published 28-gene Dataset 1 signature must not be confused with these four LOCO-selected panels.

**Private input:** `runs/native_quality_readonly/20261108_uploaded_results.zip`, whose vote tables were computed directly from native R objects. **Private outputs:** `runs/tentacles_nmin_sensitivity/20261108T113000Z/`. The notebook saves aggregate outputs to this folder, and patient predictions must remain private. This notebook reads and validates the already completed run rather than silently overwriting it.


In [ ]:
from pathlib import Path
import json
import pandas as pd
root = Path.cwd().resolve()
assert (root / 'data/validated/ibd_dataset_sample_aligned.zip').exists(), 'Open notebook from project root'
run = root/'runs/tentacles_nmin_sensitivity/20261108T113000Z'
audit = root/'runs/native_quality_readonly/20261108_uploaded_results.zip'
assert audit.is_file(), 'Native read-only audit ZIP missing; keep it private'
if not (run/'report.json').is_file():
    from src.cmps355.tentacles_nmin_sensitivity import main
    main(root,audit,run)  # computes from the original frozen native vote table, no native classifier retraining
report = json.loads((run/'report.json').read_text())
assert report['status']=='PASS_NMIN5_VS_6_PAIRED_LOCO_SENSITIVITY'
print('Sensitivity status:',report['status'])
print('Dataset SHA:',report['dataset_sha256'])
print('Input native audit SHA:',report['audit_zip_sha256'])


In [ ]:
metrics=pd.read_csv(run/'heldout_metrics_by_threshold.csv')
macro=pd.read_csv(run/'macro_summary.csv')
deltas=pd.read_csv(run/'paired_deltas.csv')
assert set(metrics.threshold)=={5,6}
assert len(metrics)==8 and metrics.groupby('threshold').heldout.nunique().eq(4).all()
assert (metrics.n_genes>0).all()
assert (deltas.n_genes_5>=deltas.n_genes_6).all()
assert abs(macro.loc[macro.threshold==5,'mean_auroc'].iloc[0]-0.6964239)<1e-5
print('Per-fold results:')
display(metrics[['heldout','threshold','n_genes','n_test','auroc','balanced_accuracy']])
print('Macro results:')
display(macro)
print('Paired n.min=6 minus n.min=5:')
display(deltas[['heldout','delta_auroc','delta_balanced_accuracy','delta_auroc_ci_lower','delta_auroc_ci_upper']])
